# InsectVision -- cascade training (Colab, free T4 GPU)

Trains both stages of the detection cascade and exports them to single-file
ONNX for the serve-time app (which uses ONNX Runtime only -- no torch, no
OpenCV -- to fit Render's free tier). Produces three files to download at the
end: `detector.onnx`, `classifier_13cls.onnx`, `config/species.json` (updated
with the real accuracy achieved, not a target).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then
`Runtime -> Restart session` if you just changed it.

**Two independent halves:**
1. **Classifier (EfficientNet-B0)** -- required. Trains on the Kaggle
   folder-per-class insect dataset **plus a 13th `other` class of non-insect
   images you supply** (section 5). Without `other`, the softmax is a forced
   choice between insects and a photo of a tree comes back as "beetle 56%";
   with it, the model has somewhere honest to put such images.
2. **Detector (YOLOv8n)** -- optional. Needs a bounding-box dataset from
   Roboflow, which the classifier's data doesn't have. Skip this whole
   section (leave the placeholder values below) and the app runs fine in
   **classifier-only mode** -- you just type a count instead of it
   auto-counting. You can always come back and add the detector later.
   **Already have a good `detector.onnx`?** Skip sections 7-8 and keep it.

## 1. GPU check

Fails loudly on purpose — training either model on CPU here would take far too long to be worth starting.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> "
        "Hardware accelerator -> GPU (T4), then Runtime -> Restart session, "
        "and re-run this notebook from the top."
    )
print(f"GPU OK: {torch.cuda.get_device_name(0)}")

## 1b. Mount Google Drive (recommended)

Detector training below can take 2-4 hours on the free tier, and Colab
sessions *can* disconnect on their own schedule regardless of what you're
doing (inactivity, time limits, or the VM just getting reclaimed).
Ultralytics saves a checkpoint after every epoch -- if that checkpoint only
lives in `/content/` and the session dies, it dies too; on Drive it
doesn't, and training can resume instead of restarting from epoch 0.

Skip this cell (set `USE_DRIVE = False`) if you'd rather not grant Drive
access -- everything still works, you just lose partial progress on a
disconnect during the detector section.

In [ ]:
import os

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_DIR = "/content/drive/MyDrive/insectvision_runs"
else:
    RUNS_DIR = "/content/runs"

os.makedirs(RUNS_DIR, exist_ok=True)
print(f"Checkpoints will be saved under: {RUNS_DIR}")

## 2. Install dependencies

In [ ]:
# Plain (CPU) onnxruntime on purpose, not onnxruntime-gpu: the export
# verification step below deliberately forces CPUExecutionProvider so it
# matches Render's CPU-only serve environment exactly, so there's no reason
# to pull in the larger CUDA-enabled package just for that check.
!pip install -q ultralytics onnx onnxruntime kaggle roboflow scikit-learn pyyaml

## 3. Get the project code

The notebook needs `src/`, `scripts/` and `config/` from the project: it runs
the real `scripts/prepare_data.py`, reads the class list from
`config/species.json`, and -- at export time -- imports the app's own
`src/classifier_onnx.py` to prove the served preprocessing matches training.

**Default: clone from GitHub** (`CODE_SOURCE = "github"`). Nothing to upload;
it pulls the `main` branch of the project repository, so make sure your
latest code is pushed.

**Fallback: upload a zip** (`CODE_SOURCE = "upload"`). From PowerShell inside
`insectvision/`:

```powershell
Compress-Archive -Path src, scripts, config -DestinationPath insectvision_code.zip -Force
```

In [ ]:
CODE_SOURCE = "github"          # "github" | "upload"
GITHUB_REPO = "https://github.com/Numbu-bit/InsectVision.git"
GITHUB_BRANCH = "retrain-13class"   # merge to "main" together with the trained model

import io, json, shutil, subprocess, sys, zipfile
from pathlib import Path

# Wipe any leftover copy from a previous (possibly failed) attempt in this
# same session, rather than silently merging old and new files.
target = Path("/content/insectvision")
if target.exists():
    shutil.rmtree(target)

if CODE_SOURCE == "github":
    print(f"Cloning {GITHUB_REPO} ({GITHUB_BRANCH})...")
    res = subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(target)],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError("git clone failed:\n" + res.stderr)
    # The repo also holds models/*.onnx (~30 MB) -- harmless, but not needed here.
    shutil.rmtree(target / "models", ignore_errors=True)
    commit = subprocess.run(["git", "-C", str(target), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
    print(f"Checked out commit {commit}")

elif CODE_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    print(f"Received {len(uploaded)} file(s): " + ", ".join(f"{n} ({len(b)} bytes)" for n, b in uploaded.items()))
    if len(uploaded) != 1:
        raise RuntimeError(
            f"Expected exactly one zip, got {len(uploaded)}. If the file picker "
            "showed a stale filename from an earlier attempt, restart the "
            "runtime (Runtime -> Restart session) and re-run from the top.")
    zip_name, zip_bytes = next(iter(uploaded.items()))
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = zf.namelist()
        print(f"Zip contains {len(names)} entries. First few:")
        for n in names[:8]:
            print(f"  {n}")
        zf.extractall(target)
else:
    raise ValueError(f"CODE_SOURCE must be 'github' or 'upload', got {CODE_SOURCE!r}")

sys.path.insert(0, str(target))

species_json = target / "config" / "species.json"
if not species_json.exists():
    found = sorted(str(p.relative_to(target)) for p in target.rglob("*"))
    raise FileNotFoundError(
        f"{species_json} not found.\n"
        f"Got {len(found)} entries instead:\n" + "\n".join(found[:40]) +
        "\n\nFor a zip: make sure you zipped src/, scripts/ and config/ directly "
        "(so config/species.json is at the zip's root), not the insectvision/ "
        "folder itself.")

species_cfg = json.loads(species_json.read_text())
class_names = species_cfg["class_names"]
taxon_status = species_cfg["taxon_status"]
print(f"\nLoaded {len(class_names)} classes:")
for taxon in class_names:
    print(f"  {taxon:<20} {taxon_status.get(taxon, 'reject class' if taxon == species_cfg.get('reject_class') else 'pest')}")

## 4. Kaggle authentication

This CLI version uses a pasted API token, not the classic browser login —
OAuth's browser redirect can't reach back to a headless Colab VM. Get a
token at **kaggle.com/settings/api** ("Create New Token"), paste it below
(hidden input, not saved anywhere but this session).

In [ ]:
import os
from getpass import getpass

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
token = getpass("Paste your Kaggle API token: ").strip()
token_path = os.path.join(kaggle_dir, "access_token")
with open(token_path, "w") as f:
    f.write(token)
os.chmod(token_path, 0o600)
print("Kaggle token saved for this session.")

## 5. Download and prepare the classifier dataset

Same script and defaults as Step 2 locally -- plus the **negative images**
that become the `other` class.

### What goes in `other`

Photos that contain **no insect** but that a user might plausibly point the
camera at: leaves, bark, grass, soil, sky, hands and fingers, tools, walls,
fabric, paper, screenshots/diagrams, blurry frames of nothing, pets, birds.
Aim for **500-1,000** images, as varied as possible -- diversity matters far
more than count.

### Three ways to supply them -- pick one in the next cell

| `NEGATIVES_SOURCE` | What happens | When to use |
|---|---|---|
| `"kaggle"` (default) | Downloads three public Kaggle datasets with your token (outdoor scenes, everyday objects/people/flowers, plant leaves), filters out anything insect-named, and samples ~350 from each | You have a Kaggle token and want a zero-effort run |
| `"upload"` | Prompts for a `negatives.zip` from your computer | You collected your own negatives |
| `"drive"` | Copies a folder from Google Drive | Same, but reusable across sessions |

Any folder layout works -- the prep script searches recursively and treats
everything as one class.

**Highest-value negatives (optional extra):** run the *detector* on
non-insect photos and save the regions it boxes -- those crops are exactly
what the classifier will be handed at serve time in cascade mode. Add them
to whichever source you use.

In [ ]:
# --- Non-insect ("other") images ---------------------------------------
NEGATIVES_SOURCE = "kaggle"     # "kaggle" | "upload" | "drive"
NEGATIVES_DRIVE_DIR = "/content/drive/MyDrive/insectvision_negatives"   # only for "drive"

MIN_NEGATIVES = 300             # fewer than this and 'other' will be too narrow to generalise
ALLOW_NO_NEGATIVES = False      # True ONLY to deliberately reproduce the old closed-set 12-class model

# Kaggle sources. Each entry: (dataset slug, how many images to sample).
# Deliberately three different KINDS of non-insect image so 'other' learns
# "not an insect" rather than "a landscape". Edit freely -- any public
# folder-of-images dataset works; a dataset that fails to download is
# reported and skipped, and MIN_NEGATIVES is still enforced at the end.
KAGGLE_NEGATIVE_SETS = [
    ("puneet6060/intel-image-classification", 350),   # buildings, forest, glacier, mountain, sea, street (~350 MB)
    ("prasunroy/natural-images", 350),                # airplane, car, cat, dog, flower, fruit, motorbike, person (~350 MB)
    ("abdallahalidev/plantvillage-dataset", 350),     # close-up plant leaves, healthy and diseased (large, ~2 GB)
]
# Folder names containing any of these words are skipped, so a general
# animal/object dataset can never smuggle insects into the negative class.
INSECT_WORDS = {"insect", "spider", "butterfly", "bee", "ant", "beetle", "moth", "bug", "fly",
                "wasp", "worm", "snail", "slug", "caterpillar", "grasshopper", "cricket", "ladybird",
                "ladybug", "mosquito", "dragonfly", "mantis", "cockroach", "termite", "aphid", "weevil",
                "ragno", "farfalla"}   # animals10 uses Italian folder names (spider, butterfly)

import io, random, re, shutil, subprocess, zipfile
from pathlib import Path

NEGATIVES_DIR = Path("/content/data/negatives")
if NEGATIVES_DIR.exists():
    shutil.rmtree(NEGATIVES_DIR)
NEGATIVES_DIR.mkdir(parents=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _looks_insecty(path: Path) -> bool:
    # Whole-word match on the path's tokens, with plural tolerance -- a plain
    # substring test would match "ant" inside "plantvillage" and throw away
    # every leaf image.
    tokens = {t for part in path.parts for t in re.split(r"[^a-z]+", part.lower()) if t}
    for t in tokens:
        if t in INSECT_WORDS or (t.endswith("s") and t[:-1] in INSECT_WORDS) or (t.endswith("ies") and t[:-3] + "y" in INSECT_WORDS):
            return True
    return False


if NEGATIVES_SOURCE == "kaggle":
    rng = random.Random(42)
    for slug, n_sample in KAGGLE_NEGATIVE_SETS:
        name = slug.split("/")[-1]
        raw_dir = Path("/content/data/neg_raw") / name
        print(f"\n[{slug}] downloading...")
        res = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(raw_dir), "--unzip", "-q"],
                             capture_output=True, text=True)
        if res.returncode != 0:
            print(f"  !! download failed -- skipping this dataset. Kaggle said:\n{(res.stdout + res.stderr).strip()[-600:]}")
            continue
        pool = [p for p in raw_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS
                and not _looks_insecty(p.relative_to(raw_dir))]
        rng.shuffle(pool)
        picked = pool[:n_sample]
        out = NEGATIVES_DIR / name
        out.mkdir(parents=True, exist_ok=True)
        for i, p in enumerate(picked):
            shutil.copy2(p, out / f"{name}_{i:04d}{p.suffix.lower()}")
        print(f"  {len(pool)} usable images found, sampled {len(picked)} -> {out}")
        shutil.rmtree(raw_dir, ignore_errors=True)   # free disk; the sample is all we keep

elif NEGATIVES_SOURCE == "upload":
    from google.colab import files
    print("Upload negatives.zip (a folder of non-insect images, any layout):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        if not fname.lower().endswith(".zip"):
            raise RuntimeError(f"Expected a .zip, got {fname}")
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            zf.extractall(NEGATIVES_DIR)

elif NEGATIVES_SOURCE == "drive":
    src = Path(NEGATIVES_DRIVE_DIR)
    if not src.exists():
        raise FileNotFoundError(f"{src} not found -- is Drive mounted (USE_DRIVE = True)?")
    shutil.copytree(src, NEGATIVES_DIR, dirs_exist_ok=True)

else:
    raise ValueError(f"NEGATIVES_SOURCE must be 'kaggle', 'upload' or 'drive', got {NEGATIVES_SOURCE!r}")

negatives = [p for p in NEGATIVES_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
print(f"\n{len(negatives)} negative images ready under {NEGATIVES_DIR}")

if len(negatives) < MIN_NEGATIVES:
    if not ALLOW_NO_NEGATIVES:
        raise RuntimeError(
            f"Only {len(negatives)} negative images (need >= {MIN_NEGATIVES}). A thin 'other' class "
            "learns 'these specific photos' instead of 'not an insect' and the tree->beetle problem "
            "comes straight back. Add another Kaggle dataset / more variety, or set "
            "ALLOW_NO_NEGATIVES = True to knowingly train the old closed-set 12-class model instead.")
    print("!!! Proceeding WITHOUT a usable 'other' class -- the model will be closed-set. !!!")
    shutil.rmtree(NEGATIVES_DIR)  # prepare_data.py treats a missing folder as 'no negatives'

In [ ]:
!kaggle datasets download -d vencerlanz09/agricultural-pests-image-dataset \
    -p /content/data/raw --unzip -q

CLASSIFY_DIR = Path("/content/data/classify")

!python /content/insectvision/scripts/prepare_data.py \
    --source /content/data/raw \
    --negatives /content/data/negatives \
    --out {CLASSIFY_DIR} \
    --species-config /content/insectvision/config/species.json \
    --max-per-class 400 --val-fraction 0.15 --seed 42

# prepare_data.py rewrote species.json (class list incl. 'other', reject_class,
# classifier_onnx filename) -- reload it so everything below sees the truth.
species_cfg = json.loads(open("/content/insectvision/config/species.json").read())
class_names = species_cfg["class_names"]
REJECT_CLASS = species_cfg.get("reject_class")

print(f"\n{len(class_names)} classes: {class_names}")
print(f"reject_class = {REJECT_CLASS!r}")
if REJECT_CLASS is None and not ALLOW_NO_NEGATIVES:
    raise RuntimeError("species.json has no reject_class -- prepare_data.py did not find the negatives. "
                       "Check the previous cell's output.")

## 6. Train the classifier (EfficientNet-B0, 13 classes)

Transfer-learned from ImageNet weights. Design choices, and why:

- **Class-weighted cross-entropy (inverse frequency)** instead of the earlier
  `WeightedRandomSampler`. Both correct class imbalance; using both at once
  double-corrects (a class that's sampled 3x as often *and* weighted 3x in the
  loss is effectively 9x). With `other` potentially 2x the size of any insect
  class, that double-correction would have pushed `other` *down* -- the
  opposite of what's wanted. Loss weights are the version that's explicit and
  auditable (they're printed).
- **Label smoothing 0.1**, kept: it stops the model saturating at 99.9% on
  everything, which is what makes the 0.75 confirmation threshold meaningful.
- **RandomErasing (p=0.4)**, kept: occlusion robustness -- an insect partly
  behind a leaf shouldn't flip the answer.
- **Model selection on macro-F1**, not accuracy, so no class can hide behind
  the others.

After training, the next cell prints a full per-class report and a
confusion matrix, and **gates the export**: any class under 0.80 F1, or
`other` recall under 0.85, or the insect-only macro-F1 dropping under 0.90
(the 12-class model scored 0.944), blocks the export unless you override.

In [ ]:
import torch
import torch.nn as nn
from collections import Counter
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = species_cfg.get("classifier_input_size", 224)

# Class order -- ImageFolder sorts folder names alphabetically, and index i
# here IS model output i. species.json's class_names is written the same way
# by prepare_data.py and asserted equal below. For the standard 12 insects +
# 'other' that order is:
#    0 ants   1 bees   2 beetle   3 catterpillar   4 earthworms   5 earwig
#    6 grasshopper   7 moth   8 other   9 slug   10 snail   11 wasp   12 weevil
# (Note 'other' sits in the MIDDLE at index 8, not at the end -- anything
# that assumes "last index = reject class" is wrong.)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    # REVIEW: hue jitter is deliberately small (0.05). Colour is a primary cue
    # for several classes (bees vs wasps, ladybird-type beetles), so a larger
    # hue shift would teach the model to ignore exactly the feature it needs.
    # Brightness/contrast/saturation at 0.3 model lighting, which SHOULD be
    # ignored. Reduce further if bees<->wasp confusion shows up in the matrix.
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.15)),
])
# The serve-time preprocessing in src/classifier_onnx.py mirrors THIS
# transform exactly (Resize(1.14x) -> CenterCrop -> ImageNet normalise); the
# export cell verifies the two agree on real images.
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(CLASSIFY_DIR / "train"), train_tf)
val_ds = datasets.ImageFolder(str(CLASSIFY_DIR / "val"), val_tf)

# Class index <-> taxon name must match species.json exactly, or the served
# app would silently report the wrong species for a correct prediction.
assert train_ds.classes == species_cfg["class_names"], (
    f"Class order mismatch.\ndataset:  {train_ds.classes}\n"
    f"species.json: {species_cfg['class_names']}")
assert val_ds.classes == train_ds.classes, "train/ and val/ have different class folders"
NUM_CLASSES = len(train_ds.classes)
REJECT_IDX = train_ds.class_to_idx[REJECT_CLASS] if REJECT_CLASS else None

print(f"{NUM_CLASSES} classes (index -> name):")
for i, name in enumerate(train_ds.classes):
    print(f"  {i:2d}  {name}{'   <- reject class' if i == REJECT_IDX else ''}")

# --- class weights: inverse frequency, normalised to mean 1.0 -------------
# w_c = N / (K * n_c). Mean 1.0 keeps the loss on the same scale as unweighted
# CE so the learning rate doesn't need retuning.
counts = Counter(train_ds.targets)
n_total = sum(counts.values())
class_weights = torch.tensor(
    [n_total / (NUM_CLASSES * counts[i]) for i in range(NUM_CLASSES)], dtype=torch.float32)
# Knob for the reject class only. 1.0 = pure inverse frequency. If the
# evaluation cell reports low 'other' recall (non-insects still leaking into
# insect classes), raise this to 1.5-2.0 and retrain; if real insects start
# being rejected, lower it.
OTHER_WEIGHT_BOOST = 1.0
if REJECT_IDX is not None:
    class_weights[REJECT_IDX] *= OTHER_WEIGHT_BOOST

print("\ntrain images / loss weight per class:")
for i, name in enumerate(train_ds.classes):
    print(f"  {name:<14} {counts[i]:>5}   w={class_weights[i]:.3f}")

BATCH = 32
# shuffle=True, NOT WeightedRandomSampler: the loss weights above are the
# balancing mechanism now (see the markdown cell for why not both).
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)

classifier_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
classifier_model.classifier[1] = nn.Linear(classifier_model.classifier[1].in_features, NUM_CLASSES)
classifier_model = classifier_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
optimiser = torch.optim.AdamW(classifier_model.parameters(), lr=3e-4, weight_decay=0.01)

EPOCHS = 30
WARMUP = 3
warmup_sched = torch.optim.lr_scheduler.LinearLR(optimiser, 0.1, 1.0, total_iters=WARMUP)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=max(1, EPOCHS - WARMUP))
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimiser, [warmup_sched, cosine_sched], milestones=[WARMUP])


def evaluate(model, loader):
    """Returns (preds, targets) over a loader -- shared by the epoch loop and
    the final report so both measure exactly the same thing."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device)).argmax(1).cpu().numpy()
            preds.extend(out.tolist())
            targets.extend(y.numpy().tolist())
    return preds, targets


best_f1, best_state, patience, stale = 0.0, None, 8, 0

for epoch in range(EPOCHS):
    classifier_model.train()
    running = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(classifier_model(x), y)
        loss.backward()
        optimiser.step()
        running += loss.item()
    scheduler.step()

    preds, targets = evaluate(classifier_model, val_dl)
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    print(f"epoch {epoch + 1:3d}/{EPOCHS}  train_loss={running / max(1, len(train_dl)):.4f}  val_macro_f1={macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1, stale = macro_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in classifier_model.state_dict().items()}
    else:
        stale += 1
        if stale >= patience:
            print(f"early stopping at epoch {epoch + 1}")
            break

classifier_model.load_state_dict(best_state)
classifier_model.eval()
print(f"\nBest validation macro-F1 during training: {best_f1:.4f}")

In [ ]:
# --- Full evaluation of the best checkpoint + export gates ----------------
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score

preds, targets = evaluate(classifier_model, val_dl)
names = train_ds.classes

classifier_macro_f1 = float(f1_score(targets, preds, average="macro", zero_division=0))
report = classification_report(targets, preds, target_names=names, zero_division=0, output_dict=True)
per_class_f1 = {n: float(report[n]["f1-score"]) for n in names}

# Insect-only macro-F1: the like-for-like number to compare against the
# 12-class model's 0.944 (which had no 'other' to score on).
specimen_idx = [i for i, n in enumerate(names) if n != REJECT_CLASS]
classifier_specimen_macro_f1 = float(f1_score(
    targets, preds, labels=specimen_idx, average="macro", zero_division=0))
reject_recall = float(report[REJECT_CLASS]["recall"]) if REJECT_CLASS else None
reject_precision = float(report[REJECT_CLASS]["precision"]) if REJECT_CLASS else None

print(f"Validation macro-F1 (all {len(names)} classes): {classifier_macro_f1:.4f}")
print(f"Validation macro-F1 (insects only, vs 0.944 baseline): {classifier_specimen_macro_f1:.4f}")
if REJECT_CLASS:
    print(f"'{REJECT_CLASS}' recall={reject_recall:.3f}  precision={reject_precision:.3f}")
print("\nPer-class report:")
print(classification_report(targets, preds, target_names=names, zero_division=0, digits=3))

# Confusion matrix, rows = true class, cols = predicted. The 'other' ROW
# shows what non-insects get mistaken for (the tree->beetle failure); the
# 'other' COLUMN shows real insects being wrongly rejected.
cm = confusion_matrix(targets, preds, labels=list(range(len(names))))
try:
    import pandas as pd
    short = [n[:6] for n in names]
    print("\nConfusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(cm, index=names, columns=short).to_string())
except ImportError:
    print("\nConfusion matrix (rows=true, cols=pred):\n", cm)

if REJECT_CLASS:
    r = names.index(REJECT_CLASS)
    leaks = sorted(((cm[r, j], names[j]) for j in range(len(names)) if j != r), reverse=True)[:3]
    rejected = sorted(((cm[i, r], names[i]) for i in range(len(names)) if i != r), reverse=True)[:3]
    print(f"\nNon-insects most often mis-called as an insect: {leaks}")
    print(f"Insects most often wrongly rejected as '{REJECT_CLASS}': {rejected}")

# --- Gates ------------------------------------------------------------------
MIN_CLASS_F1 = 0.80
MIN_REJECT_RECALL = 0.85
MIN_SPECIMEN_MACRO_F1 = 0.90   # 12-class baseline was 0.944; a dip to ~0.90 is acceptable, below is not

gate_failures = []
for n, f in per_class_f1.items():
    if f < MIN_CLASS_F1:
        gate_failures.append(f"class '{n}' F1 {f:.3f} < {MIN_CLASS_F1}")
if REJECT_CLASS and reject_recall < MIN_REJECT_RECALL:
    gate_failures.append(f"'{REJECT_CLASS}' recall {reject_recall:.3f} < {MIN_REJECT_RECALL} -- "
                         "non-insects are still leaking into insect classes. More/varied negatives, "
                         "or raise OTHER_WEIGHT_BOOST and retrain.")
if classifier_specimen_macro_f1 < MIN_SPECIMEN_MACRO_F1:
    gate_failures.append(f"insect-only macro-F1 {classifier_specimen_macro_f1:.3f} < {MIN_SPECIMEN_MACRO_F1} -- "
                         "a large dip from the 0.944 baseline. Something is wrong (bad negatives that "
                         "look like insects? class-order mismatch? too few epochs?) -- investigate "
                         "before exporting.")

TRAINING_GATES_PASSED = not gate_failures
if TRAINING_GATES_PASSED:
    print("\nOK: All export gates passed.")
else:
    print("\n" + "!" * 72)
    print("WARNING: EXPORT GATES FAILED:")
    for g in gate_failures:
        print("   - " + g)
    print("The export cell will refuse to run unless EXPORT_DESPITE_WARNINGS = True there.")
    print("!" * 72)

## 7. Detector training data (Roboflow) -- OPTIONAL, off by default

**You already have a trained `detector.onnx` (mAP@0.50 = 0.964) and this
notebook run is about the classifier.** Leave `TRAIN_DETECTOR = False` in
the next cell and sections 7-8 are skipped entirely; the existing detector
keeps working and its metric is preserved in `species.json`.

Set `TRAIN_DETECTOR = True` only if you want to retrain it. That needs a
Roboflow API key and 2-4 hours on a free T4.

---


Using **FAIR-D v2.5** (workspace `fairdevice-htj8d`, project `fair-d_v2.5`,
version 2): 21,426 images across 39 fine-grained insect-trap classes,
CC BY 4.0. Found via Roboflow Universe search and verified directly
(image/label counts, `data.yaml`) before being wired in below -- not
guessed. Want a different dataset instead? Just edit the three values in
the next cell.

**Why the detector's 39 classes don't need to match the classifier's 12
taxa:** at serve time, the detector's only job is finding *where* insects
are -- species identity always comes from the classifier running on each
cropped region, never from the detector's own class prediction (see
`app.py`, Step 4). So a dataset with a completely different class list
still works fine. By default this notebook collapses every box to one
class, `insect`, before training -- a class-agnostic localiser is an
easier, more data-efficient task, and with 21k+ images merged into a
single class there's plenty of data for it to converge well. Set
`COLLAPSE_TO_SINGLE_CLASS = False` below to keep the original 39 classes
instead, if you'd rather.

**Heads up on training time:** this is a big dataset -- ~1,170 batches/epoch
at batch=16. Expect roughly 3-6 minutes/epoch on a free T4, so the reduced
40-epoch budget below (down from a generic 80) is still likely 2-4 hours.
Free Colab sessions can disconnect on their own schedule regardless of your
activity; if training stops partway, `runs/detect/insectvision_detector/weights/last.pt`
holds the latest checkpoint and `model.train(resume=True)` (pointed at that
run) picks back up rather than restarting from scratch.

In [ ]:
TRAIN_DETECTOR = False   # you already have models/detector.onnx -- set True ONLY to retrain it (2-4 h on a T4)

ROBOFLOW_WORKSPACE = "fairdevice-htj8d"  # FAIR-D v2.5 -- 21,426 images, 39 fine classes, CC BY 4.0
ROBOFLOW_PROJECT = "fair-d_v2.5"
ROBOFLOW_VERSION = 2
COLLAPSE_TO_SINGLE_CLASS = True

RUN_DETECTOR_TRAINING = TRAIN_DETECTOR and ROBOFLOW_WORKSPACE != "PASTE_WORKSPACE_HERE"
if not RUN_DETECTOR_TRAINING:
    print("Detector training is OFF -- skipping sections 7-8.")
    print("The existing models/detector.onnx keeps being used (cascade mode), and its")
    print("recorded detector_map50 is carried over into species.json unchanged.")
else:
    print(f"Will train detector on {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}")

In [ ]:
if RUN_DETECTOR_TRAINING:
    from getpass import getpass
    from roboflow import Roboflow

    rf_key = getpass("Paste your Roboflow API key (roboflow.com -> Settings -> API key): ").strip()
    rf = Roboflow(api_key=rf_key)
    rf_project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    rf_dataset = rf_project.version(ROBOFLOW_VERSION).download("yolov8", location="/content/data/detect")
    print(f"Downloaded to {rf_dataset.location}")

In [ ]:
import yaml
from pathlib import Path

DETECT_DIR = Path("/content/data/detect")

if RUN_DETECTOR_TRAINING:
    data_yaml_path = DETECT_DIR / "data.yaml"
    detect_cfg = yaml.safe_load(data_yaml_path.read_text())

    if COLLAPSE_TO_SINGLE_CLASS:
        for split in ("train", "valid", "val", "test"):
            label_dir = DETECT_DIR / split / "labels"
            if not label_dir.exists():
                continue
            for label_file in label_dir.glob("*.txt"):
                fixed = []
                for line in label_file.read_text().splitlines():
                    parts = line.split()
                    if not parts:
                        continue
                    parts[0] = "0"  # collapse every class to a single "insect" class
                    fixed.append(" ".join(parts))
                label_file.write_text("\n".join(fixed) + ("\n" if fixed else ""))
        detect_cfg["names"] = ["insect"]
        detect_cfg["nc"] = 1
        print("Collapsed all boxes to a single 'insect' class.")

    # Rewrite as absolute paths -- Roboflow's exported relative paths are
    # relative to wherever ultralytics happens to resolve them from at
    # train time, which isn't reliably this notebook's cwd.
    val_split = "valid" if (DETECT_DIR / "valid").exists() else "val"
    detect_cfg["path"] = str(DETECT_DIR)
    detect_cfg["train"] = "train/images"
    detect_cfg["val"] = f"{val_split}/images"
    data_yaml_path.write_text(yaml.safe_dump(detect_cfg))
    print(detect_cfg)

## 8. Train the detector (YOLOv8n)

In [ ]:
if RUN_DETECTOR_TRAINING:
    from ultralytics import YOLO

    yolo_model = YOLO("yolov8n.pt")
    yolo_model.train(
        data=str(DETECT_DIR / "data.yaml"),
        epochs=40,  # generous for 21k images; early stopping (patience=20) will cut this short once it plateaus
        imgsz=640,
        batch=16,
        device=0,
        optimizer="SGD",
        lr0=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,
        patience=20,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=20.0, scale=0.5, fliplr=0.5, flipud=0.3,
        mosaic=1.0, close_mosaic=10,
        erasing=0.4,
        project=f"{RUNS_DIR}/detect",
        name="insectvision_detector",
    )
    detector_metrics = yolo_model.val()
    detector_map50 = float(detector_metrics.box.map50)
    print(f"\nDetector mAP@0.50 = {detector_map50:.4f}")
else:
    detector_map50 = None

### If you got disconnected during detector training

Re-run the setup cells (GPU check, Drive mount, dependency install, code
upload, Kaggle auth + data prep are all fast) up through the Roboflow
config cell, **then run this cell instead of the training cell above** --
it resumes from the last saved checkpoint rather than starting over. Skip
it entirely if training completed without interruption.

In [ ]:
if RUN_DETECTOR_TRAINING:
    from ultralytics import YOLO

    checkpoint = f"{RUNS_DIR}/detect/insectvision_detector/weights/last.pt"
    yolo_model = YOLO(checkpoint)
    yolo_model.train(resume=True)

    detector_metrics = yolo_model.val()
    detector_map50 = float(detector_metrics.box.map50)
    print(f"\nDetector mAP@0.50 = {detector_map50:.4f}")

## 9. Export both models to single-file ONNX

Newer export paths sometimes switch to external-data storage past a size
threshold (an `.onnx.data` sidecar next to the `.onnx` file), which is easy
to lose track of if only the `.onnx` file gets copied out. `fold_external_data`
collapses everything back into one self-contained file -- harmlessly, even
if no sidecar was produced.

The classifier export is checked three ways before it's trusted:
1. torch vs ONNX Runtime logits on the same random input (max drift < 1e-3);
2. the ONNX output width equals `len(class_names)` -- the file is named by
   class count (`classifier_13cls.onnx`) and the serve-time loader
   re-checks this, so a 12-class file can never be served against a
   13-entry class list;
3. **serve-path parity**: the app's own `src/classifier_onnx.py` (uploaded in
   section 3) is run on real validation images and must agree with the torch
   model + `val_tf` pipeline. This is the check that catches a preprocessing
   mismatch between training and production -- the classic silent failure.

In [ ]:
import shutil
import numpy as np
import onnx
import onnxruntime as ort

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)


def fold_external_data(onnx_path: str) -> None:
    m = onnx.load(onnx_path, load_external_data=True)
    onnx.save(m, onnx_path, save_as_external_data=False)


def assert_close(name: str, torch_out: np.ndarray, onnx_out: np.ndarray, tol: float = 1e-3) -> None:
    drift = float(np.abs(torch_out - onnx_out).max())
    print(f"{name}: torch vs onnx max drift = {drift:.2e} (tolerance {tol:.0e})")
    assert drift < tol, f"{name} ONNX export drifted too far from the torch model ({drift:.2e} >= {tol:.0e})"

In [ ]:
# --- classifier ---
EXPORT_DESPITE_WARNINGS = False   # set True to export a model that failed the section-6 gates (not recommended)

if not TRAINING_GATES_PASSED and not EXPORT_DESPITE_WARNINGS:
    raise RuntimeError("Export blocked: the evaluation gates failed (see the report above). "
                       "Fix the training data / settings, or set EXPORT_DESPITE_WARNINGS = True to override.")

# Named by class count so it can't be confused with the old 12-class file.
classifier_onnx_name = f"classifier_{NUM_CLASSES}cls.onnx"
classifier_onnx_path = str(MODELS_DIR / classifier_onnx_name)

classifier_model.eval().cpu()

# BatchNorm health: channels whose running variance collapsed to ~0 make
# eval-mode BN a huge multiplier, which amplifies float noise (and makes the
# model brittle on anything slightly off-distribution). Rare with real data
# and a full training run; a symptom of a tiny/degenerate training set.
_bn_var = torch.cat([m.running_var.flatten() for m in classifier_model.modules()
                     if isinstance(m, torch.nn.BatchNorm2d)])
_collapsed = int((_bn_var < 1e-6).sum())
print(f"BatchNorm channels: {len(_bn_var)}, running_var min={_bn_var.min():.2e}, collapsed(<1e-6)={_collapsed}")
if _collapsed:
    print(f"WARNING: {_collapsed} BatchNorm channels have collapsed variance -- REVIEW before deploying.")

dummy_classifier_input = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)  # shape/tracing only; parity is checked on real images below
export_kwargs = dict(
    opset_version=13,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    do_constant_folding=True,
)
# torch >= 2.9 defaults to the new "dynamo" exporter, which needs the
# onnxscript package and only supports opset >= 18. The classic TorchScript
# exporter is what we want (opset 13, dynamic_axes); older torch versions
# don't know the `dynamo` argument at all, hence the fallback.
try:
    torch.onnx.export(classifier_model, dummy_classifier_input, classifier_onnx_path, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(classifier_model, dummy_classifier_input, classifier_onnx_path, **export_kwargs)
fold_external_data(classifier_onnx_path)
sess = ort.InferenceSession(classifier_onnx_path, providers=["CPUExecutionProvider"])

# (1) numeric parity torch vs ONNX Runtime -- on a batch of REAL, normalised
# validation images, compared on softmax probabilities.
#   * Real images, not torch.randn: unnormalised noise is far outside the
#     input distribution the BatchNorm statistics describe, and was measured
#     to inflate torch/ORT drift 1000x versus the same model on real photos
#     while telling us nothing about how the model behaves on actual inputs.
#   * Probabilities, not raw logits with a fixed absolute tolerance: logit
#     scale is arbitrary, so "1e-3 absolute" means something different for
#     every model. Probabilities are what the app thresholds on.
#   * A batch of 8 also proves the dynamic batch axis works (export traced 1).
_parity_idx = np.random.default_rng(1).choice(len(val_ds), size=min(8, len(val_ds)), replace=False)
_batch = torch.stack([val_ds[int(i)][0] for i in _parity_idx])
with torch.no_grad():
    torch_probs = torch.softmax(classifier_model(_batch), dim=1).numpy()
onnx_logits = sess.run(None, {"input": _batch.numpy()})[0]
onnx_probs = np.exp(onnx_logits - onnx_logits.max(1, keepdims=True)); onnx_probs /= onnx_probs.sum(1, keepdims=True)
prob_drift = float(np.abs(torch_probs - onnx_probs).max())
print(f"classifier: torch vs onnx on {len(_batch)} real val images -- max |dprob| = {prob_drift:.2e}, "
      f"argmax agree = {(torch_probs.argmax(1) == onnx_probs.argmax(1)).all()}")
assert prob_drift < 1e-4 and (torch_probs.argmax(1) == onnx_probs.argmax(1)).all(),     "ONNX export does not reproduce the torch model on real images -- do not deploy this file."

# (2) output width == class count
assert onnx_logits.shape == (len(_batch), NUM_CLASSES), f"ONNX output shape {onnx_logits.shape}, expected ({len(_batch)}, {NUM_CLASSES})"
print(f"ONNX output width {NUM_CLASSES} == len(class_names) OK")

# (3) serve-path parity: the app's real preprocessing + ONNX Runtime vs torch + val_tf
from PIL import Image
from src.classifier_onnx import ClassifierOnnx   # from the code uploaded in section 3

serve_clf = ClassifierOnnx(classifier_onnx_path, class_names, IMG_SIZE)  # also re-validates the class count
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(val_ds.samples), size=min(96, len(val_ds.samples)), replace=False)
agree, max_prob_drift = 0, 0.0
with torch.no_grad():
    for i in sample_idx:
        path, _ = val_ds.samples[i]
        img = Image.open(path).convert("RGB")
        torch_probs = torch.softmax(classifier_model(val_tf(img)[None]), dim=1)[0].numpy()
        serve_probs = serve_clf.probabilities(img)
        agree += int(torch_probs.argmax() == serve_probs.argmax())
        max_prob_drift = max(max_prob_drift, float(np.abs(torch_probs - serve_probs).max()))
agreement = agree / len(sample_idx)
print(f"serve-path parity on {len(sample_idx)} val images: argmax agreement={agreement:.3f}, "
      f"max |dprob|={max_prob_drift:.4f}")
assert agreement >= 0.98 and max_prob_drift < 0.02, (
    "The app's src/classifier_onnx.py preprocessing does NOT match the training val_tf pipeline. "
    "Do not deploy -- fix one of them so they agree.")
print("Serve-path parity OK -- src/classifier_onnx.py reproduces the training-time preprocessing.")

classifier_model.to(device)  # move back in case any later cell is re-run

In [ ]:
# --- detector ---
detector_onnx_path = str(MODELS_DIR / "detector.onnx")

if RUN_DETECTOR_TRAINING:
    exported_path = yolo_model.export(format="onnx", imgsz=640, opset=17, simplify=True)
    shutil.copy(exported_path, detector_onnx_path)
    fold_external_data(detector_onnx_path)

    underlying = yolo_model.model.eval().cpu()
    dummy_detector_input = torch.rand(1, 3, 640, 640)
    with torch.no_grad():
        torch_raw = underlying(dummy_detector_input)
        if isinstance(torch_raw, (list, tuple)):
            torch_raw = torch_raw[0]
        torch_raw = torch_raw.numpy()

    sess = ort.InferenceSession(detector_onnx_path, providers=["CPUExecutionProvider"])
    onnx_input_name = sess.get_inputs()[0].name
    onnx_raw = sess.run(None, {onnx_input_name: dummy_detector_input.numpy()})[0]
    assert_close("detector", torch_raw, onnx_raw)
else:
    print("Detector training was skipped (see section 7) -- no detector.onnx produced.")
    print("The app will run in classifier-only mode, which is fully functional.")

## 10. Write real results into species.json

In [ ]:
species_cfg["classifier_onnx"] = f"models/{classifier_onnx_name}"
species_cfg["classifier_macro_f1"] = classifier_macro_f1
species_cfg["classifier_specimen_macro_f1"] = classifier_specimen_macro_f1
species_cfg["classifier_per_class_f1"] = per_class_f1
species_cfg["classifier_reject_recall"] = reject_recall
species_cfg["detector_map50"] = detector_map50 if RUN_DETECTOR_TRAINING else species_cfg.get("detector_map50")

species_config_out = "/content/insectvision/config/species.json"
with open(species_config_out, "w") as f:
    json.dump(species_cfg, f, indent=2)
    f.write("\n")

print(f"classifier_onnx               = {species_cfg['classifier_onnx']}")
print(f"classifier_macro_f1           = {classifier_macro_f1:.4f}")
print(f"classifier_specimen_macro_f1  = {classifier_specimen_macro_f1:.4f}")
print(f"classifier_reject_recall      = {reject_recall}")
print(f"detector_map50                = {species_cfg['detector_map50']}")
print(f"\nWrote {species_config_out}")

## 11. Download your files

Place `classifier_13cls.onnx` (and `detector.onnx`, if you trained one) in
`insectvision/models/`, and `species.json` in `insectvision/config/`
(overwriting the copy there), then commit and push -- Step 7 covers the
actual Render deploy.

The old `models/classifier.onnx` (12-class) can be deleted once the new file
is in place: `species.json` now points at the new name, and the app refuses
to load a model whose class count doesn't match `class_names` anyway.

In [ ]:
from google.colab import files

files.download(classifier_onnx_path)
if RUN_DETECTOR_TRAINING:
    files.download(detector_onnx_path)
files.download(species_config_out)

## 12. Recovery: re-export a detector from a Drive checkpoint

Use this section if a downloaded `detector.onnx` turns out to have been
exported from the wrong in-memory model (symptom: `detector_map50` comes
back as exactly `0.0`, and the ONNX output shape has 80-something classes
instead of 1 -- that's the untrained base `yolov8n.pt`, not your
fine-tuned checkpoint). This happens if `yolo_model` gets reassigned
between the training cell and the export cell finishing, e.g. from
re-running cells out of order after a disconnect.

**Self-contained** -- doesn't depend on any other cells having run in this
session except "1. GPU check" and "2. Install dependencies" (re-run those
two first if this is a fresh runtime). Loads the checkpoint directly from
Drive rather than trusting whatever `yolo_model` currently holds, and
prints its class count as an explicit sanity check *before* doing anything
else with it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_PATH = "/content/drive/MyDrive/insectvision_runs/detect/insectvision_detector/weights/best.pt"

from ultralytics import YOLO
yolo_model = YOLO(CHECKPOINT_PATH)
print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
print(f"nc={yolo_model.model.nc}, names={yolo_model.model.names}")

assert yolo_model.model.nc == 1, (
    f"Expected the fine-tuned single-class ('insect') checkpoint but got "
    f"nc={yolo_model.model.nc} ({yolo_model.model.names}) -- this isn't "
    "the right file. Don't continue with this checkpoint."
)
print("\nnc == 1 confirmed -- this is the real fine-tuned checkpoint.")

### Re-download the validation split to measure a real mAP@0.50

Needs your Roboflow API key again (nothing from before was cached).

In [ ]:
from getpass import getpass
from pathlib import Path
import yaml
from roboflow import Roboflow

rf_key = getpass("Paste your Roboflow API key: ").strip()
rf = Roboflow(api_key=rf_key)
rf_dataset = (rf.workspace("fairdevice-htj8d").project("fair-d_v2.5")
             .version(2).download("yolov8", location="/content/data/detect"))

DETECT_DIR = Path("/content/data/detect")
data_yaml_path = DETECT_DIR / "data.yaml"
detect_cfg = yaml.safe_load(data_yaml_path.read_text())

for split in ("train", "valid", "val", "test"):
    label_dir = DETECT_DIR / split / "labels"
    if not label_dir.exists():
        continue
    for label_file in label_dir.glob("*.txt"):
        fixed = []
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            parts[0] = "0"  # same collapse-to-single-class as the original run
            fixed.append(" ".join(parts))
        label_file.write_text("\n".join(fixed) + ("\n" if fixed else ""))

val_split = "valid" if (DETECT_DIR / "valid").exists() else "val"
detect_cfg["names"] = ["insect"]
detect_cfg["nc"] = 1
detect_cfg["path"] = str(DETECT_DIR)
detect_cfg["train"] = "train/images"
detect_cfg["val"] = f"{val_split}/images"
data_yaml_path.write_text(yaml.safe_dump(detect_cfg))

detector_metrics = yolo_model.val(data=str(data_yaml_path))
detector_map50 = float(detector_metrics.box.map50)
print(f"\nReal detector mAP@0.50 = {detector_map50:.4f}")

### Export + verify (same procedure as section 9, detector only)

In [ ]:
import shutil
import numpy as np
import onnx
import onnxruntime as ort
import torch
from pathlib import Path

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)


def fold_external_data(onnx_path: str) -> None:
    m = onnx.load(onnx_path, load_external_data=True)
    onnx.save(m, onnx_path, save_as_external_data=False)


detector_onnx_path = str(MODELS_DIR / "detector.onnx")
exported_path = yolo_model.export(format="onnx", imgsz=640, opset=17, simplify=True)
shutil.copy(exported_path, detector_onnx_path)
fold_external_data(detector_onnx_path)

underlying = yolo_model.model.eval().cpu()
dummy = torch.rand(1, 3, 640, 640)
with torch.no_grad():
    torch_raw = underlying(dummy)
    if isinstance(torch_raw, (list, tuple)):
        torch_raw = torch_raw[0]
    torch_raw = torch_raw.numpy()

sess = ort.InferenceSession(detector_onnx_path, providers=["CPUExecutionProvider"])
onnx_raw = sess.run(None, {sess.get_inputs()[0].name: dummy.numpy()})[0]

print(f"exported output shape: {onnx_raw.shape}  (expect (1, 5, 8400) for a 1-class model)")
assert onnx_raw.shape[1] == 5, f"expected 5 rows (4 box + 1 class), got {onnx_raw.shape[1]} -- still wrong"

drift = float(np.abs(torch_raw - onnx_raw).max())
print(f"torch vs onnx max drift = {drift:.2e} (tolerance 1e-3)")
assert drift < 1e-3, f"drift too high: {drift:.2e}"
print("\nExport verified correct.")

### Download the corrected file

Just `detector.onnx` this time -- your classifier was already correct, no
need to redo it. Overwrite `insectvision/models/detector.onnx` with this,
and update the `detector_map50` value printed above into your local
`config/species.json` by hand (or hand both to me and I'll do it).

In [ ]:
from google.colab import files

files.download(detector_onnx_path)
print(f"\ndetector_map50 = {detector_map50:.4f}  <-- record this in config/species.json")